# Week 24, Day 6: Forge: Week 24 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w024/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("1. sqrt(900) =", np.sqrt(900))
print("2. sigma sqrt(tau / 2) =", 0.5 * np.sqrt(8 / 2))
print("3.", round(1 / (1 + np.exp(-2 * 1 * 1.5)), 4))

### Exercise: Part A: mean-driven and fluctuation-driven firing

Simulate 50 neurons at once for each regime (tau = 20 ms, theta = 15 mV, refractory 2 ms, dt = 0.1 ms, 5 s). Collect every neuron's interspike intervals, and store the pooled CV in `cv[name]` and the mean rate in Hz in `rate[name]`.

In [ ]:
tau, theta, tref, dt = 20.0, 15.0, 2.0, 0.1
r_a = np.random.default_rng(1964)
cv, rate = {}, {}
for name, RI, sig in (("mean", 20.0, 1.0), ("fluct", 8.0, 5.0)):
    n = 50; V = np.zeros(n); refr = np.zeros(n); last = np.full(n, np.nan); isis = []; n_sp = 0
    for k in range(int(5000 / dt)):
        free = refr <= 0
        V[free] += ...                            # Euler-Maruyama step of the OU voltage
        refr -= dt
        spk = V >= theta
        t = k * dt
        for i in np.where(spk)[0]:
            if not np.isnan(last[i]): isis.append(t - last[i])
            last[i] = t
        n_sp += spk.sum(); V[spk] = 0.0; refr[spk] = tref
    isis = np.array(isis)
    cv[name] = ...
    rate[name] = n_sp / (n * 5.0)
print({k: round(float(v), 3) for k, v in cv.items()}, " CV;", {k: round(float(v), 1) for k, v in rate.items()}, " Hz")

In [ ]:
import numpy as np
def test_regular():
    "mean-driven firing (RI = 20, sigma = 1) is regular: CV below 0.3"
    assert cv["mean"] < 0.3

def test_irregular():
    "fluctuation-driven firing (RI = 8, sigma = 5) is irregular: CV above 0.7"
    assert cv["fluct"] > 0.7

def test_rates():
    "both regimes fire at least 5 Hz"
    assert rate["mean"] > 5 and rate["fluct"] > 5

test_regular()
test_irregular()
test_rates()
print("All checks passed")

### Exercise: Part B: fitting the drift diffusion model

Simulate the observer (1,000 trials per strength, dt = 0.001) to get `acc_obs` and `rt_obs`. Then, on a grid of k from 0.5 to 4 and a from 0.5 to 2 (steps of 0.01), compute the predicted accuracy and mean time from Day 3's formulas, and choose the pair minimising the sum over strengths of (accuracy error / 0.02)² + (time error / (0.05 · observed time))². Store them in `k_fit` and `a_fit`.

In [ ]:
cohs = np.array([0.1, 0.25, 0.5])
r_b = np.random.default_rng(2007)
acc_obs, rt_obs = [], []
for c in cohs:
    mu = 2.0 * c
    x = np.zeros(1000); th = np.zeros(1000); won = np.zeros(1000, bool); live = np.ones(1000, bool); t = 0.0
    while live.any():
        t += 0.001
        x[live] += mu * 0.001 + np.sqrt(0.001) * r_b.normal(size=live.sum())
        up, dn = live & (x >= 1), live & (x <= -1)
        won[up] = True; th[up | dn] = t; live &= ~(up | dn)
    acc_obs.append(won.mean()); rt_obs.append(th.mean())
acc_obs, rt_obs = np.array(acc_obs), np.array(rt_obs)

K, A_ = np.meshgrid(np.arange(0.5, 4.0001, 0.01), np.arange(0.5, 2.0001, 0.01), indexing="ij")
loss = np.zeros_like(K)
for c, acc_o, rt_o in zip(cohs, acc_obs, rt_obs):
    mu = K * c
    acc_p = 1 / (1 + np.exp(-2 * mu * A_))
    rt_p = ...
    loss += ...
i, j = np.unravel_index(np.argmin(loss), loss.shape)
k_fit, a_fit = K[i, j], A_[i, j]
print("observed accuracy", acc_obs.round(3), " mean time", rt_obs.round(3)); print(f"recovered k = {k_fit:.2f}, a = {a_fit:.2f}")

In [ ]:
import numpy as np
def test_k():
    "the recovered k is within 10% of 2"
    assert abs(k_fit - 2) / 2 < 0.10

def test_a():
    "the recovered bound is within 10% of 1"
    assert abs(a_fit - 1) < 0.10

test_k()
test_a()
print("All checks passed")

### Exercise: Part C: Kramers escape

For D = 0.6, 0.5 and 0.4, start 300 particles at x = −1 and simulate (dt = 0.005) until each first reaches x = +1. Store the mean times in `T_esc`, and the slope of log T_esc against 1/D in `slope_C`.

In [ ]:
Ds = np.array([0.6, 0.5, 0.4])
U_grad = lambda x: 8 * x * (x ** 2 - 1)
r_c = np.random.default_rng(1940)
T_esc = []
for D in Ds:
    x = -np.ones(300); t_hit = np.zeros(300); live = np.ones(300, bool); k = 0
    while live.any():
        k += 1
        x[live] += ...
        h = live & (x >= 1); t_hit[h] = k * 0.005; live &= ~h
    T_esc.append(t_hit.mean())
T_esc = np.array(T_esc)
slope_C = ...
print("mean escape times", T_esc.round(1), f";  slope of log T against 1/D: {slope_C:.2f} (barrier 2)")

In [ ]:
import numpy as np
def test_order():
    "less noise means longer escape times"
    assert np.all(np.diff(T_esc) > 0)

def test_slope():
    "log escape time against 1/D has slope within 0.3 of the barrier height 2"
    assert abs(slope_C - 2) < 0.3

test_order()
test_slope()
print("All checks passed")